In [7]:
# Load the env file
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
# Load openai (note : .env has to contain exactly 'OPENAI_API_KEY' and so should Openai API Dashboard)
from openai import OpenAI
openai_client = OpenAI()


In [ ]:
# Dataset : FAQ
# send get request for courses details
import requests
courses_url = "https://datatalks.club/faq/json/courses.json"
#by default, the http connection is kept alive for subsequent requests
response = requests.get(courses_url)
courses_raw = response.json() #get the response in json
# print(courses_raw)
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [ ]:
#form urls for the faqs for each of the courses and store in a list.
documents = []
base_url = "https://datatalks.club/faq"
for course in courses_raw:
    final_url = base_url + course["path"]
    course_response = requests.get(final_url)
    course_response.raise_for_status() #raises http error if one occurred
    course_faq = course_response.json() #each response is a list of jsons
    # documents.append(course_faq) #if append is used, each element of documents would be a list of jsons. [[{},..], [{},..]]. Length would be 6.
    documents.extend(course_faq) #if extend is used, each element of documents is a json. [{}, {}]. Length would be 1000+.
print(len(documents))
#this list now contains all of our information for faqs for the courses. We will put this in knowledge base amd this will act as the documents for RAG.



1342


In [17]:
documents[400]

{'id': '53f31cd33c',
 'course': 'data-engineering-zoomcamp',
 'section': 'Workshop 1 - dlthub',
 'question': 'Why might some API response fields be missing after a dlt pipeline run, and how can I fix it?',
 'answer': 'Root cause:\n- dlt infers column types from actual data in the current load. If a column contains only NULL values in that load, dlt cannot infer its type and the column will not be materialized in the destination table.\n\nFixes:\n1) Provide explicit type hints using the columns argument in the @dlt.resource decorator.\n\nExample:\n```python\n@dlt.resource(columns={\n    "rate_code": "STRING",\n    "mta_tax": "FLOAT64"\n})\ndef api_records():\n    # your data loading logic here\n    return fetch_api()\n```\n\n2) Ensure at least one non-null value exists for that column during ingestion.\n   - If possible, modify the API payload or add a preprocessing step to emit a non-null value for the field (e.g., default values).\n   - Example pre-processing (in Python):\n```python\n

In [18]:
# Build index and Search
#building index
from minsearch import Index
index = Index(text_fields=["question", "section", "answer"],
              keyword_fields=["course"])
index.fit(documents)


In [23]:
#searching by giving some arguments for boosting and filtering
# question : str = "I just discovered the course. Can I join now?"
# search_results = index.search(question, 
#                               boost_dict= {"question" : 2.0, "section" : 0.5},
#                               filter_dict = {"course" : "llm-zoomcamp"},
#                               num_results=5
#                               )
# # search_results
# [result["question"] for result in search_results] #the retrieved questions (top-5)

In [31]:
def rag():
    #1. search
    #2. build prompt
    #3. return generated result of llm
    pass

# creating a method for search
def search(question, course = "llm-zoomcamp"):
    search_results = index.search(question, 
                              boost_dict= {"question" : 2.0, "section" : 0.5},
                              filter_dict = {"course" : course},
                              num_results=5
                              )
    return search_results

In [33]:
#1. search
question : str = "I just discovered the course. Can I join now?"
search_results = search(question)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [66]:
# Final prompt = s/m prompt(instructions) + user prompt
# But we send to llm as two and not one. So keep it seperate
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

# USER_PROMPT_TEMPLATE = """
# Question:
# {}

# Context:
# {}
# """


In [ ]:
# step 1 : build user prompt
# step 1a : build context from results of the search (this does not have the user question, and has only the search results)
# context will have the k results together
def build_context(search_results) -> str:
    lines = []
    for result in search_results:
        lines.append(result["section"]) #directly adding the section
        lines.append("Q: " + result["question"]) #prepending Q: at start and then adding to list
        lines.append("A: " + result["answer"]) #prepending A: at start and then adding to list
        lines.append("") #empty line
    return "\n".join(lines).strip() #return back a string, so using join

In [47]:
# print(build_context(search_results))

In [60]:
#step 1b : form the user prompt (question + context)
def build_user_prompt(question, search_results) -> str:
    context : str = build_context(search_results)
    USER_PROMPT_TEMPLATE = """
Question:
{}

Context:
{}
    """.format(question, context)
    return USER_PROMPT_TEMPLATE.strip()

    

In [ ]:
user_prompt = build_user_prompt(question, search_results)
print(user_prompt)

#so now we have user_prompt and INSTRUCTIONS in 2 seperate variables
# user_prompt, INSTRUCTIONS


Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode a